# Prompt Engineering for Sinhala Food Ordering

**Technique:** Prompt Engineering (Advanced AI module requirement)

This notebook demonstrates and compares three prompting strategies:
1. **Zero-shot** — No examples, just a system prompt
2. **Few-shot** — Provide example conversations in the prompt
3. **Chain-of-Thought (CoT)** — Ask the model to reason step-by-step before answering

We measure which strategy produces the best food-ordering responses from the base Gemma 4 model (before fine-tuning).

In [ ]:
import json
import re
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# We'll call a model via HuggingFace Inference API (free tier)
# OR use the local fine-tuned model if available
import requests

HF_TOKEN = os.getenv('HF_TOKEN', '')   # your HuggingFace token
BASE_MODEL = 'google/gemma-3-4b-it'   # base model (no fine-tuning)

DATA_DIR = Path('../data')

print('Setup complete.')

## Helper: call the model

In [ ]:
def call_hf_inference(messages: list[dict], model_id: str = BASE_MODEL, max_tokens: int = 300) -> str:
    """
    Call HuggingFace Inference API (free tier).
    Falls back to a mock response if no token or quota exceeded.
    """
    if not HF_TOKEN:
        return '[HF_TOKEN not set — using mock response]'

    url = f'https://api-inference.huggingface.co/models/{model_id}/v1/chat/completions'
    headers = {'Authorization': f'Bearer {HF_TOKEN}', 'Content-Type': 'application/json'}
    payload = {
        'model': model_id,
        'messages': messages,
        'max_tokens': max_tokens,
        'temperature': 0.7,
    }
    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=30)
        resp.raise_for_status()
        return resp.json()['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f'[Error: {e}]'


# Quick smoke-test
test = call_hf_inference([{'role': 'user', 'content': 'Hello in Sinhala'}])
print('Model response:', test)

## Strategy 1: Zero-Shot Prompting

In [ ]:
ZERO_SHOT_SYSTEM = """You are an AI food ordering assistant for Colombo Kitchen restaurant in Sri Lanka.
You must understand Sinhala, English, and mixed Sinhala-English messages.
Collect the customer's order (items, quantity, spice level), delivery address, then confirm."""

def zero_shot(user_message: str) -> str:
    messages = [
        {'role': 'system', 'content': ZERO_SHOT_SYSTEM},
        {'role': 'user', 'content': user_message},
    ]
    return call_hf_inference(messages)


test_inputs = [
    'චිකන් කොට්ටු දෙකක් ඕන medium spice',
    'machang 3 hot mutton kottu denna',
    'vegetarian options monawada?',
]

print('Zero-Shot Results:')
print('=' * 60)
zs_results = []
for inp in test_inputs:
    resp = zero_shot(inp)
    zs_results.append({'input': inp, 'response': resp})
    print(f'Input: {inp}')
    print(f'Response: {resp[:250]}')
    print()
    time.sleep(1)

## Strategy 2: Few-Shot Prompting

In [ ]:
FEW_SHOT_EXAMPLES = """
Example 1:
Customer: චිකන් කොට්ටු ඕන
Assistant: Chicken Kottu - Rs.450. කීයක් ඕනේද? Spice level (mild/medium/hot)?

Example 2:
Customer: 2 hot
Assistant: Chicken Kottu x2 (hot) - Rs.900. Delivery address?

Example 3:
Customer: Kottawa, main street 12
Assistant: Confirm:\nChicken Kottu x2 (hot) - Rs.900\nDelivery - Rs.150\nTotal - Rs.1050\nKottawa, Main Street 12\n\nConfirm?

Example 4:
Customer: vegetarian options monawada?
Assistant: Vegetarian options:\n- Vegetable Kottu - Rs.300\n- Egg Fried Rice - Rs.300\n- Vegetable Noodles - Rs.280\n- Rice & Curry - Rs.350\n\nKamathi monawada?
"""

FEW_SHOT_SYSTEM = ZERO_SHOT_SYSTEM + "\n\nHere are example conversations:\n" + FEW_SHOT_EXAMPLES

def few_shot(user_message: str) -> str:
    messages = [
        {'role': 'system', 'content': FEW_SHOT_SYSTEM},
        {'role': 'user', 'content': user_message},
    ]
    return call_hf_inference(messages)


print('Few-Shot Results:')
print('=' * 60)
fs_results = []
for inp in test_inputs:
    resp = few_shot(inp)
    fs_results.append({'input': inp, 'response': resp})
    print(f'Input: {inp}')
    print(f'Response: {resp[:250]}')
    print()
    time.sleep(1)

## Strategy 3: Chain-of-Thought (CoT) Prompting

In [ ]:
COT_SYSTEM = ZERO_SHOT_SYSTEM + """

Before responding, think step by step:
1. Identify the customer intent (order/inquiry/confirm/cancel/other)
2. Extract entities: food items, quantities, spice level, address
3. Check what information is missing
4. Compose a response in the customer's language

Format your internal reasoning as:
<think>
Intent: ...
Entities: ...
Missing: ...
</think>

Then give only the customer-facing response (without the <think> block)."""

def chain_of_thought(user_message: str) -> tuple[str, str]:
    """Returns (reasoning, final_response)."""
    messages = [
        {'role': 'system', 'content': COT_SYSTEM},
        {'role': 'user', 'content': user_message},
    ]
    full = call_hf_inference(messages, max_tokens=400)

    # Extract reasoning and final response
    think_match = re.search(r'<think>(.*?)</think>', full, re.DOTALL)
    reasoning = think_match.group(1).strip() if think_match else ''
    response = re.sub(r'<think>.*?</think>', '', full, flags=re.DOTALL).strip()
    return reasoning, response


print('Chain-of-Thought Results:')
print('=' * 60)
cot_results = []
for inp in test_inputs:
    reasoning, resp = chain_of_thought(inp)
    cot_results.append({'input': inp, 'reasoning': reasoning, 'response': resp})
    print(f'Input: {inp}')
    if reasoning:
        print(f'Reasoning: {reasoning[:200]}')
    print(f'Response: {resp[:250]}')
    print()
    time.sleep(1)

## Comparison & Scoring

In [ ]:
# Score each response on keyword presence
EXPECTED_KEYWORDS = [
    ['Chicken Kottu', 'Rs.', 'quantity', 'spice'],
    ['Mutton Kottu', 'hot', '3', 'address'],
    ['Vegetable', 'Rs.', 'available'],
]

def score_response(response: str, keywords: list) -> float:
    r = response.lower()
    hits = sum(1 for kw in keywords if kw.lower() in r)
    return hits / len(keywords)

results_data = []
for i, (inp, kws) in enumerate(zip(test_inputs, EXPECTED_KEYWORDS)):
    zs_score = score_response(zs_results[i]['response'], kws)
    fs_score = score_response(fs_results[i]['response'], kws)
    cot_score = score_response(cot_results[i]['response'], kws)
    results_data.append({
        'Input': inp[:40] + '...',
        'Zero-Shot': zs_score,
        'Few-Shot': fs_score,
        'Chain-of-Thought': cot_score,
    })

df = pd.DataFrame(results_data)
print('Keyword Score Comparison (1.0 = all keywords present):')
print(df.to_string(index=False))
print(f'\nMean scores:')
print(f'  Zero-Shot:        {df["Zero-Shot"].mean():.2f}')
print(f'  Few-Shot:         {df["Few-Shot"].mean():.2f}')
print(f'  Chain-of-Thought: {df["Chain-of-Thought"].mean():.2f}')

In [ ]:
# Visualise
means = {
    'Zero-Shot': df['Zero-Shot'].mean(),
    'Few-Shot': df['Few-Shot'].mean(),
    'Chain-of-Thought': df['Chain-of-Thought'].mean(),
}

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(means.keys(), means.values(), color=colors, width=0.5)
ax.set_ylabel('Mean Keyword Score')
ax.set_title('Prompt Engineering Comparison — Sinhala Food Ordering')
ax.set_ylim(0, 1.1)
for bar, val in zip(bars, means.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('../data/prompt_engineering_comparison.png', dpi=150)
plt.show()
print('Chart saved.')